# 01 - Download Partial Safe Subset: `gr1_arms_waist.TrayToPlate`

Notebook này chỉ tải **1 subset** để tránh một notebook ôm quá nhiều data và làm đầy disk Kaggle.

- Download mode mặc định: `data_meta_ego_chunks`, tức là tải `meta/`, `data/`, và chỉ video `ego_view` ở các chunk được chọn.
- Không tải full video mặc định vì subset video rất lớn và dễ tràn disk Kaggle.
- Nếu hết disk, sửa riêng notebook này sang `DOWNLOAD_MODE = "data_meta_only"`.
- `PIPELINE_SAFE_FOR_NOTEBOOK02 = True`.

Gợi ý vận hành:

1. Chạy notebook này trên Kaggle CPU.
2. Nếu output thành công, save output thành Kaggle Dataset.
3. Add nhiều output download notebook vào Notebook 02.
4. Notebook 02 sẽ merge các file `selected_subsets_for_notebook02.json`.

In [1]:
!pip install -q huggingface_hub tqdm

In [2]:
from pathlib import Path
import json, os, shutil, time, traceback

try:
    from kaggle_secrets import UserSecretsClient
except Exception:
    UserSecretsClient = None

from huggingface_hub import HfApi, snapshot_download

REPO_ID = "nvidia/PhysicalAI-Robotics-GR00T-X-Embodiment-Sim"
SUBSET_NAME = "gr1_arms_waist.TrayToPlate"
PRIORITY_GROUP = "p1_gr1_safe"
PIPELINE_SAFE_FOR_NOTEBOOK02 = True

LOCAL_DIR = Path("/kaggle/working/gr00t_x_embodiment_sim")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

# Disk Kaggle không đủ cho full video; mặc định tải meta/data + một ít video ego_view để đủ chạy VLM encode.
DOWNLOAD_MODE = "data_meta_ego_chunks"  # "full_subset" | "data_meta_ego_chunks" | "data_meta_only"
CAMERA_KEY = "observation.images.ego_view"
VIDEO_CHUNKS = [0, 1, 2]
CLEAN_HF_CACHE_AFTER_DOWNLOAD = True

REPORT_PATH = Path("/kaggle/working/download_report_gr1_arms_waist_TrayToPlate.json")
SELECTED_PATH = Path("/kaggle/working/selected_subsets_for_notebook02.json")

print("SUBSET_NAME:", SUBSET_NAME)
print("PRIORITY_GROUP:", PRIORITY_GROUP)
print("PIPELINE_SAFE_FOR_NOTEBOOK02:", PIPELINE_SAFE_FOR_NOTEBOOK02)
print("DOWNLOAD_MODE:", DOWNLOAD_MODE)
print("LOCAL_DIR:", LOCAL_DIR)

SUBSET_NAME: gr1_arms_waist.TrayToPlate
PRIORITY_GROUP: p1_gr1_safe
PIPELINE_SAFE_FOR_NOTEBOOK02: True
DOWNLOAD_MODE: data_meta_ego_chunks
LOCAL_DIR: /kaggle/working/gr00t_x_embodiment_sim


In [3]:
def get_token():
    # Không hard-code HF token. Dùng Kaggle Secret HF_TOKEN hoặc env HF_TOKEN.
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    if UserSecretsClient is not None:
        try:
            return UserSecretsClient().get_secret("HF_TOKEN")
        except Exception as exc:
            print("WARN: cannot read Kaggle Secret HF_TOKEN:", exc)
    return None

def disk_free_gb(path="/kaggle/working"):
    return shutil.disk_usage(path).free / (1024 ** 3)

def folder_size_gb(path):
    path = Path(path)
    if not path.exists():
        return 0.0
    total = 0
    for p in path.rglob("*"):
        if p.is_file():
            try:
                total += p.stat().st_size
            except OSError:
                pass
    return total / (1024 ** 3)

def count_files(path, suffix=None):
    path = Path(path)
    if not path.exists():
        return 0
    if suffix is None:
        return sum(1 for p in path.rglob("*") if p.is_file())
    return sum(1 for p in path.rglob(f"*{suffix}") if p.is_file())

def clean_hf_cache():
    # Full subset co the tao cache lon; xoa cache sau download de output gon hon.
    for p in [LOCAL_DIR / ".cache", Path("/kaggle/working/.cache/huggingface")]:
        if p.exists():
            shutil.rmtree(p, ignore_errors=True)

def allow_patterns():
    if DOWNLOAD_MODE == "full_subset":
        return [f"{SUBSET_NAME}/**"]
    patterns = [f"{SUBSET_NAME}/meta/**", f"{SUBSET_NAME}/data/**"]
    if DOWNLOAD_MODE == "data_meta_ego_chunks":
        for chunk_id in VIDEO_CHUNKS:
            patterns.append(f"{SUBSET_NAME}/videos/chunk-{chunk_id:03d}/{CAMERA_KEY}/**")
    return patterns

token = get_token()
api = HfApi(token=token)

# Kiem tra subset co ton tai tren Hugging Face truoc khi tai.
try:
    root_items = list(api.list_repo_tree(repo_id=REPO_ID, repo_type="dataset", path_in_repo="", recursive=False))
    available = {x.path for x in root_items if getattr(x, "path", None)}
    if SUBSET_NAME not in available:
        raise ValueError(f"Subset not found in HF dataset: {SUBSET_NAME}")
except Exception as exc:
    print("WARN: repo listing failed or subset not listed. Will still try snapshot_download.")
    print(repr(exc))

print("HF token:", "OK" if token else "NONE/PUBLIC")
print("Free disk before:", round(disk_free_gb(), 3), "GB")
print("Allow patterns:")
for pat in allow_patterns():
    print(" -", pat)

HF token: OK
Free disk before: 19.502 GB
Allow patterns:
 - gr1_arms_waist.TrayToPlate/meta/**
 - gr1_arms_waist.TrayToPlate/data/**
 - gr1_arms_waist.TrayToPlate/videos/chunk-000/observation.images.ego_view/**
 - gr1_arms_waist.TrayToPlate/videos/chunk-001/observation.images.ego_view/**
 - gr1_arms_waist.TrayToPlate/videos/chunk-002/observation.images.ego_view/**


In [4]:
subset_path = LOCAL_DIR / SUBSET_NAME
started = time.time()
status = "unknown"
err = None

try:
    snapshot_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        local_dir=str(LOCAL_DIR),
        allow_patterns=allow_patterns(),
        token=token,
        resume_download=True,
        local_dir_use_symlinks=False,
    )
    status = "downloaded"
except Exception as exc:
    status = "error"
    err = {"repr": repr(exc), "traceback": traceback.format_exc()[-4000:]}
    print("ERROR:", repr(exc))

if CLEAN_HF_CACHE_AFTER_DOWNLOAD:
    clean_hf_cache()

report = {
    "repo_id": REPO_ID,
    "subset_name": SUBSET_NAME,
    "priority_group": PRIORITY_GROUP,
    "pipeline_safe_for_notebook02": PIPELINE_SAFE_FOR_NOTEBOOK02,
    "download_mode": DOWNLOAD_MODE,
    "allow_patterns": allow_patterns(),
    "local_dir": str(LOCAL_DIR),
    "subset_path": str(subset_path),
    "status": status,
    "error": err,
    "elapsed_sec": round(time.time() - started, 2),
    "exists": subset_path.exists(),
    "size_gb": round(folder_size_gb(subset_path), 3),
    "has_data_dir": (subset_path / "data").exists(),
    "has_meta_dir": (subset_path / "meta").exists(),
    "has_videos_dir": (subset_path / "videos").exists(),
    "num_parquet_files": count_files(subset_path / "data", ".parquet"),
    "num_video_files": count_files(subset_path / "videos", ".mp4"),
    "free_gb_after": round(disk_free_gb(), 3),
}
REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching ... files: 0it [00:00, ?it/s]

HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/nvidia/PhysicalAI-Robotics-GR00T-X-Embodiment-Sim/resolve/ea7ac0b68f87da62f1e726771bba0fe74300802f/gr1_arms_waist.TrayToPlate/data/chunk-007/episode_007822.parquet
Rate limited. Waiting 34.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/nvidia/PhysicalAI-Robotics-GR00T-X-Embodiment-Sim/resolve/ea7ac0b68f87da62f1e726771bba0fe74300802f/gr1_arms_waist.TrayToPlate/data/chunk-007/episode_007821.parquet
Rate limited. Waiting 34.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/nvidia/PhysicalAI-Robotics-GR00T-X-Embodiment-Sim/resolve/ea7ac0b68f87da62f1e726771bba0fe74300802f/gr1_arms_waist.TrayToPlate/data/chunk-007/episode_007823.parquet
Rate limited. Waiting 34.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/nvidia/PhysicalAI-Robotics-GR00T-X-Embodim

{
  "repo_id": "nvidia/PhysicalAI-Robotics-GR00T-X-Embodiment-Sim",
  "subset_name": "gr1_arms_waist.TrayToPlate",
  "priority_group": "p1_gr1_safe",
  "pipeline_safe_for_notebook02": true,
  "download_mode": "data_meta_ego_chunks",
  "allow_patterns": [
    "gr1_arms_waist.TrayToPlate/meta/**",
    "gr1_arms_waist.TrayToPlate/data/**",
    "gr1_arms_waist.TrayToPlate/videos/chunk-000/observation.images.ego_view/**",
    "gr1_arms_waist.TrayToPlate/videos/chunk-001/observation.images.ego_view/**",
    "gr1_arms_waist.TrayToPlate/videos/chunk-002/observation.images.ego_view/**"
  ],
  "local_dir": "/kaggle/working/gr00t_x_embodiment_sim",
  "subset_path": "/kaggle/working/gr00t_x_embodiment_sim/gr1_arms_waist.TrayToPlate",
  "status": "downloaded",
  "error": null,
  "elapsed_sec": 1253.2,
  "exists": true,
  "size_gb": 4.329,
  "has_data_dir": true,
  "has_meta_dir": true,
  "has_videos_dir": true,
  "num_parquet_files": 10074,
  "num_video_files": 3000,
  "free_gb_after": 15.136
}


In [5]:
downloaded_ok = (
    report["status"] == "downloaded"
    and report["has_data_dir"]
    and report["has_meta_dir"]
)

selected = {
    "repo_id": REPO_ID,
    "subset_name": SUBSET_NAME,
    "priority_group": PRIORITY_GROUP,
    "download_mode": DOWNLOAD_MODE,
    "pipeline_safe_for_notebook02": PIPELINE_SAFE_FOR_NOTEBOOK02,
    "downloaded_ok": downloaded_ok,
    "pipeline_safe_subsets": [SUBSET_NAME] if downloaded_ok and PIPELINE_SAFE_FOR_NOTEBOOK02 else [],
    "downloaded_subsets": [SUBSET_NAME] if downloaded_ok else [],
    "note": "Notebook 02 will merge pipeline_safe_subsets from all added download outputs.",
}
SELECTED_PATH.write_text(json.dumps(selected, indent=2), encoding="utf-8")
Path("/kaggle/working/downloaded_subsets.txt").write_text((SUBSET_NAME + "\n") if downloaded_ok else "", encoding="utf-8")

print("Selected plan:")
print(json.dumps(selected, indent=2))

Selected plan:
{
  "repo_id": "nvidia/PhysicalAI-Robotics-GR00T-X-Embodiment-Sim",
  "subset_name": "gr1_arms_waist.TrayToPlate",
  "priority_group": "p1_gr1_safe",
  "download_mode": "data_meta_ego_chunks",
  "pipeline_safe_for_notebook02": true,
  "downloaded_ok": true,
  "pipeline_safe_subsets": [
    "gr1_arms_waist.TrayToPlate"
  ],
  "downloaded_subsets": [
    "gr1_arms_waist.TrayToPlate"
  ],
  "note": "Notebook 02 will merge pipeline_safe_subsets from all added download outputs."
}


## Nếu notebook này hết disk

Sửa riêng notebook này:

```python
DOWNLOAD_MODE = "data_meta_ego_chunks"
VIDEO_CHUNKS = [0, 1, 2]
```

Nếu vẫn đầy disk, giảm video xuống:

```python
VIDEO_CHUNKS = [0]
```

Nếu vẫn đầy disk:

```python
DOWNLOAD_MODE = "data_meta_only"
```

Sau đó rerun notebook subset này, không cần sửa các notebook subset khác.